# 483 – SAT Concordance Validator
Robust validator for SAT ↔ OSM ↔ Wikidata.

In [1]:
from datetime import date
import time
import requests
import pandas as pd

TODAY=date.today().strftime("%Y%m%d")
PREFIX="483"

CONCORDANCE_URL="https://map.stockholmarchipelagotrail.com/data/geojson/poi-concordance.json"

OVERPASS_ENDPOINTS=[
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

SPARQL_ENDPOINT="https://query.wikidata.org/sparql"

session=requests.Session()
session.headers.update({
    "User-Agent":"SAT-Concordance-Validator/1.0 (https://github.com/salgo60/Stockholm_Archipelago_Trail)",
    "Accept":"application/json"
})


In [2]:
def get_json(url, params=None, headers=None, timeout=180, retries=5):
    for attempt in range(retries):
        r=session.get(url, params=params, headers=headers, timeout=timeout)
        if r.status_code==429:
            wait=int(r.headers.get("Retry-After","30"))
            print(f"429 - waiting {wait}s")
            time.sleep(wait)
            continue
        if r.status_code>=500:
            time.sleep(2**attempt)
            continue
        r.raise_for_status()
        ctype=r.headers.get("Content-Type","")
        if "json" not in ctype:
            raise RuntimeError(f"Expected JSON, got {ctype}\n{r.text[:500]}")
        return r.json()
    raise RuntimeError("Too many retries")


In [3]:
def load_concordance():
    data=get_json(CONCORDANCE_URL)
    rows=[]
    for ext,sat in data["satIdOf"].items():
        p=ext.split(":")
        source=p[0]
        row={"source":source,"external_id":ext,"sat_id":sat,
             "osm_type":None,"osm_id":None,"qid":None}
        if source=="osm":
            row["osm_type"]=p[1]
            row["osm_id"]=int(p[2])
        elif source=="wikidata":
            row["qid"]=p[1]
        rows.append(row)
    return pd.DataFrame(rows)

concordance=load_concordance()
display(concordance.head())


,source,external_id,sat_id,osm_type,osm_id,qid
0,grillplatser,grillplatser:G-0254e7bf-dd61-49dc-81c4-784e9e7...,sat:poi:eb5j5,None,NaN,None
1,grillplatser,grillplatser:G-052fd3c1-9648-4339-b206-074c5d1...,sat:poi:cc37k,None,NaN,None
2,grillplatser,grillplatser:G-09fc2541-3901-4a50-a68c-5fac553...,sat:poi:fqw8k,None,NaN,None
3,grillplatser,grillplatser:G-0a3e8dd4-6364-4fbd-b6ab-1122266...,sat:poi:459zw,None,NaN,None
4,grillplatser,grillplatser:G-0f556cb2-bd14-4049-a5ad-5b95277...,sat:poi:4f4rw,None,NaN,None


In [4]:
query='''[out:json][timeout:120];
(
 nwr["ref:stockholmarchipelagotrail"];
);
out ids tags;'''

def load_osm():
    last=None
    for ep in OVERPASS_ENDPOINTS:
        try:
            data=get_json(ep,params={"data":query})
            rows=[]
            for e in data["elements"]:
                rows.append({
                    "osm_type":e["type"],
                    "osm_id":e["id"],
                    "sat_id":e["tags"].get("ref:stockholmarchipelagotrail")
                })
            return pd.DataFrame(rows)
        except Exception as ex:
            print(ep, ex)
            last=ex
    raise last

osm=load_osm()
display(osm.head())


,osm_type,osm_id,sat_id
0,node,5530966,sat:pier:badnq
1,node,5530979,sat:pier:x9t5q
2,node,268908517,sat:poi:y2cyg
3,node,268910308,sat:poi:n6mfb
4,node,268910309,sat:poi:w3rmg


In [5]:
sparql='''SELECT ?item ?sat WHERE {
  ?item wdt:P14545 ?sat .
}'''

headers={
 "Accept":"application/sparql-results+json",
 "User-Agent":session.headers["User-Agent"]
}

data=get_json(SPARQL_ENDPOINT,
              params={"query":sparql},
              headers=headers)

wd=pd.DataFrame([{
    "qid":b["item"]["value"].split("/")[-1],
    "sat_id":b["sat"]["value"]
} for b in data["results"]["bindings"]])

display(wd.head())


,qid,sat_id
0,Q134583669,sat:poi:embmb
1,Q134581015,sat:poi:kzfw6
2,Q134593750,sat:poi:2y3sb
3,Q134588307,sat:poi:dcm2b
4,Q134589066,sat:poi:e2sqb


In [6]:
conc_osm=concordance.query("source=='osm'")[["sat_id","osm_type","osm_id"]]
report_osm=conc_osm.merge(osm,on="sat_id",how="outer",suffixes=("_conc","_osm"))
report_osm.to_csv(f"{PREFIX}_concordance_vs_osm_{TODAY}.csv",index=False)

conc_wd=concordance.query("source=='wikidata'")[["sat_id","qid"]]
report_wd=conc_wd.merge(wd,on="sat_id",how="outer",suffixes=("_conc","_wd"))
report_wd.to_csv(f"{PREFIX}_concordance_vs_wikidata_{TODAY}.csv",index=False)

qs=report_wd[report_wd["qid_conc"].isna() & report_wd["qid_wd"].notna()].copy()
if not qs.empty:
    qs["QS"]=qs.apply(lambda r:f'{r["qid_wd"]}\tP14545\t"{r["sat_id"]}"',axis=1)
    qs["QS"].to_csv(f"{PREFIX}_quickstatements_add_satid_{TODAY}.tsv",
                    index=False,header=False)

print("Done")


Done
